# 03 — Feature Engineering

**Goal:** Turn the cleaned data into a fully numeric table a model can learn from.

**Plan:**
1. Simplify redundant categories
2. Engineer new features based on EDA findings
3. Encode categorical columns as numbers
4. Split into train and test sets
5. Scale numeric features (using train data only)

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/processed/telco_clean.csv")
df = df.drop(columns=["customerID"])
df["Churn"] = (df["Churn"] == "Yes").astype(int)

print("Shape:", df.shape)

Shape: (7043, 20)


## 1. Simplify redundant categories

"No internet service" / "No phone service" duplicate information already held in
InternetService and PhoneService. Collapsing them into "No" removes the redundancy
without losing information.

In [2]:
service_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]

for col in service_cols:
    df[col] = df[col].replace("No internet service", "No")

df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

print(df["TechSupport"].value_counts())

TechSupport
No     4999
Yes    2044
Name: count, dtype: int64


## 2. Engineered features

Each feature encodes an EDA finding so the model can use it directly.
None uses the Churn column, so there is no target leakage.

In [3]:
df["NumAddons"] = (df[service_cols] == "Yes").sum(axis=1)

df["HasProtection"] = ((df["OnlineSecurity"] == "Yes") |
                       (df["TechSupport"] == "Yes")).astype(int)

df["NewCustomer"] = (df["tenure"] <= 12).astype(int)

df["FiberMonthToMonth"] = ((df["InternetService"] == "Fiber optic") &
                           (df["Contract"] == "Month-to-month")).astype(int)

df["AvgMonthlySpend"] = np.where(df["tenure"] > 0,
                                 df["TotalCharges"] / df["tenure"],
                                 df["MonthlyCharges"])

df["PriceChange"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]

df[["NumAddons", "HasProtection", "NewCustomer", "FiberMonthToMonth",
    "AvgMonthlySpend", "PriceChange"]].describe().round(2)

,NumAddons,HasProtection,NewCustomer,FiberMonthToMonth,AvgMonthlySpend,PriceChange
count,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00
mean,2.04,0.42,0.31,0.30,64.76,-0.00
std,1.85,0.49,0.46,0.46,30.19,2.61
min,0.00,0.00,0.00,0.00,13.78,-18.90
25%,0.00,0.00,0.00,0.00,35.94,-1.16
50%,2.00,0.00,0.00,0.00,70.34,0.00
75%,3.00,1.00,1.00,1.00,90.17,1.15
max,6.00,1.00,1.00,1.00,121.40,19.13


## 3. Encoding categorical variables

- Yes/No columns → 1/0
- gender → IsMale (1/0)
- Multi-category columns (InternetService, Contract, PaymentMethod) → one-hot encoding

In [4]:
yes_no_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling",
               "MultipleLines"] + service_cols

for col in yes_no_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

df["IsMale"] = (df["gender"] == "Male").astype(int)
df = df.drop(columns=["gender"])

df = pd.get_dummies(df, columns=["InternetService", "Contract", "PaymentMethod"],
                    drop_first=True, dtype=int)

print("Shape:", df.shape)
print()
print(df.dtypes)

Shape: (7043, 30)

SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
MultipleLines                              int64
OnlineSecurity                             int64
OnlineBackup                               int64
DeviceProtection                           int64
TechSupport                                int64
StreamingTV                                int64
StreamingMovies                            int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
NumAddons                                  int64
HasProtection                              int64
NewCustomer                                int64
FiberMonthToMonth                          int64
A

In [5]:
df.to_csv("../data/processed/telco_features.csv", index=False)
print("Saved:", df.shape)

Saved: (7043, 30)


## 4. Train/test split

80% of customers for training, 20% held back for final evaluation.
`stratify=y` keeps the churn rate (~26.5%) the same in both sets, which matters
because the classes are imbalanced.

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Churn rate in train: {y_train.mean():.1%}")
print(f"Churn rate in test:  {y_test.mean():.1%}")

Train: (5634, 29) | Test: (1409, 29)
Churn rate in train: 26.5%
Churn rate in test:  26.5%


## 5. Scaling numeric features

Only continuous columns are scaled (0/1 columns need no scaling).
The scaler learns mean and std from the TRAIN set only, then applies the same
transformation to the test set, so no test information leaks into training.
Tree models will use the unscaled version; Logistic Regression the scaled one.

In [7]:
from sklearn.preprocessing import StandardScaler

num_cols = ["tenure", "MonthlyCharges", "TotalCharges",
            "NumAddons", "AvgMonthlySpend", "PriceChange"]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print(X_train_scaled[num_cols].describe().loc[["mean", "std"]].round(2))

      tenure  MonthlyCharges  TotalCharges  NumAddons  AvgMonthlySpend  \
mean    -0.0            -0.0           0.0        0.0             -0.0   
std      1.0             1.0           1.0        1.0              1.0   

      PriceChange  
mean         -0.0  
std           1.0  


In [8]:
import os
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.joblib")

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
X_train_scaled.to_csv("../data/processed/X_train_scaled.csv", index=False)
X_test_scaled.to_csv("../data/processed/X_test_scaled.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Saved train/test sets and scaler")

Saved train/test sets and scaler


## Summary

- 29 features after encoding and engineering; target is `Churn`
- 6 engineered features, each tied to an EDA finding, none using the target
- 80/20 stratified split (random_state=42); churn rate ~26.5% in both sets
- Scaler fitted on train only to prevent leakage; saved for the Streamlit app